# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya – Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is accessible via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed. Uncomment and run if needed.
!pip install mlcroissant

## 1. Data Loading

We load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL (from FAIR^2)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\n{metadata.description}\n\nPublished: {metadata.datePublished}\nVersion: {metadata.version}")

## 2. Data Overview

In this section, we inspect the available record sets, and for each, list their corresponding fields and columns. All are referenced by their `@id`.

If record sets are empty, list available record sets by their IDs and inspect.

In [ ]:
# Enumerate all record sets with their @id
print("Available record sets:")
record_sets = []
for rs in dataset.record_sets():
    print(f"- @id: {rs['@id']} | Name: {rs.get('name','(no name)')}")
    record_sets.append(rs['@id'])

if len(record_sets) == 0:
    print("No record sets were found in the top-level Croissant schema. Searching distributions...")
    # Sometimes record sets are found in distributions or supplementary schemas.
    print("Use dataset.distributions() to explore data files as well.")

Let's now preview the fields and columns for each available record set (using their `@id`). The output will show `@id`, name, and data type for each field.

In [ ]:
# For each record set, list the fields by @id
for rs_id in record_sets:
    print(f"\nRecord set @id: {rs_id}")
    fields = dataset.fields(rs_id)
    for field in fields:
        print(f"    Field @id: {field['@id']}")
        print(f"       name: {field.get('name')}")
        print(f"       dataType: {field.get('dataType')}")

To further understand the data, let's preview the first record of each record set available (referenced by `@id`).

In [ ]:
# Preview first record (if available) for each record set by @id
for rs_id in record_sets:
    print(f"\nFirst record from record set @id: {rs_id}")
    try:
        recs = dataset.records(record_set=rs_id)
        first = next(recs)
        print(first)
    except StopIteration:
        print("No records available.")
    except Exception as e:
        print(f"Error accessing records: {e}")

## 3. Data Extraction

Now, we load data from all record sets into pandas DataFrames for exploration and processing, referencing record sets by their `@id` throughout. Each DataFrame is accessible by its record set `@id`.

In [ ]:
# Extract records into DataFrames for each available record set
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rs_id] = df
    print(f"\nRecord set @id: {rs_id} -- {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

if not dataframes:
    print("No DataFrames extracted; please check the record set IDs and schema.")

## 4. Exploratory Data Analysis (EDA)

In this section, we process the loaded data: filter records, normalize numeric fields, categorize, group, and prepare for analysis. All fields and columns are referenced by their `@id`.

*Please adjust the `record_set_id`, `numeric_field_id`, and `group_field_id` below as appropriate for your data as found in the previous section outputs.*

In [ ]:
# === EDA Example ===
# Pick an example record set @id, numeric field @id, and group field @id from the record set info.

example_record_set_id = None
example_numeric_field_id = None
example_group_field_id = None

# Find a DataFrame with numeric columns
for rs_id, df in dataframes.items():
    if not df.empty:
        numeric_cols = df.select_dtypes(include='number').columns
        if len(numeric_cols) > 0:
            example_record_set_id = rs_id
            example_numeric_field_id = numeric_cols[0]
            # Try to guess a group field (non-numeric, with low cardinality)
            candidate_group_cols = [c for c in df.columns if (df[c].dtype == object and df[c].nunique() < max(10, 0.1*len(df)))]
            if candidate_group_cols:
                example_group_field_id = candidate_group_cols[0]
            break

print(f"Example record_set_id: {example_record_set_id}")
print(f"Example numeric_field_id: {example_numeric_field_id}")
print(f"Example group_field_id: {example_group_field_id}")

if example_record_set_id and example_numeric_field_id:
    df = dataframes[example_record_set_id]

    # Filter: select rows where numeric value > threshold
    threshold = df[example_numeric_field_id].median()
    filtered_df = df[df[example_numeric_field_id] > threshold].copy()
    print(f"\nFiltered records (where {example_numeric_field_id} > {threshold}): {filtered_df.shape[0]}")
    print(filtered_df[[example_numeric_field_id]].head())

    # Normalize the numeric field
    mean = filtered_df[example_numeric_field_id].mean()
    std = filtered_df[example_numeric_field_id].std()
    filtered_df[f"{example_numeric_field_id}_normalized"] = (filtered_df[example_numeric_field_id] - mean) / std
    print(f"\nNormalized {example_numeric_field_id}:")
    print(filtered_df[[example_numeric_field_id, f"{example_numeric_field_id}_normalized"]].head())

    # Group by group field if available
    if example_group_field_id and example_group_field_id in df.columns:
        grouped = filtered_df.groupby(example_group_field_id, dropna=False)[example_numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {example_numeric_field_id} by {example_group_field_id}:")
        print(grouped)
else:
    print("No suitable numeric column for EDA found. Please check your dataset.")

## 5. Visualization

Let's visualize the distribution of the chosen numeric field for the selected record set, and its grouping by the selected group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and example_numeric_field_id:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[example_numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {example_numeric_field_id} in record set {example_record_set_id}")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Optional: visualize group-wise mean
    if example_group_field_id and example_group_field_id in df.columns:
        group_means = df.groupby(example_group_field_id)[example_numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8, 5))
        sns.barplot(data=group_means, x=example_group_field_id, y=example_numeric_field_id)
        plt.title(f"Group-wise mean of {example_numeric_field_id} by {example_group_field_id}")
        plt.xlabel(example_group_field_id)
        plt.ylabel(f"Mean {example_numeric_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to programmatically explore and process a Croissant-structured dataset using the `mlcroissant` library:

- Accessed dataset metadata and examined available record sets and their fields (by `@id`).
- Loaded records from one or more record sets into pandas DataFrames.
- Applied basic filtering, normalization, and grouping operations on numeric fields.
- Visualized the distribution and group-wise summary of a numeric field.

Further analysis can leverage the detailed metadata and structured processing enabled by Croissant schemas and `mlcroissant` for robust, transparent, and FAIR-aligned research workflows.